# HECKTOR case explorer (Google Colab)

Load a HECKTOR case from Google Drive and visualize CT with mask overlay, slice by slice.

**Paths (Colab + Drive):**
- Cases root: `/content/drive/MyDrive/phD/phD-Petia/Trainings/Hecktor/cases/`
- Per case: `{case_id}/` with `{case_id}.nii.gz` (mask) and `{case_id}__CT.nii.gz` (CT)
- Expected volume shape: (512, 512, 91)

## 1. Mount Google Drive and config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

# HECKTOR paths on Drive
CASES_ROOT = "/content/drive/MyDrive/phD/phD-Petia/Trainings/Hecktor/cases"
case_id = "CHUM-001"
case_folder = os.path.join(CASES_ROOT, case_id)
path_ct = os.path.join(case_folder, f"{case_id}__CT.nii.gz")
path_mask = os.path.join(case_folder, f"{case_id}.nii.gz")

print("Case folder:", case_folder)
print("CT:", path_ct, "-> exists:", os.path.exists(path_ct))
print("Mask:", path_mask, "-> exists:", os.path.exists(path_mask))

## 2. Load CT and mask

In [ ]:
# Load volumes (shape 512, 512, 91)
ct_nii = nib.load(path_ct)
mask_nii = nib.load(path_mask)
ct_vol = ct_nii.get_fdata().astype(np.float32)
mask_vol = mask_nii.get_fdata().astype(np.int32)

print("CT shape:", ct_vol.shape)
print("Mask shape:", mask_vol.shape)
print("Mask unique labels:", np.unique(mask_vol))

# Normalize CT for display (1-99 percentile)
p1, p99 = np.percentile(ct_vol, (1, 99))
ct_norm = np.clip((ct_vol - p1) / (p99 - p1 + 1e-8), 0.0, 1.0)

## 3. Side-by-side: CT and CT + mask overlay (interactive slice)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

n_slices = ct_vol.shape[2]

def show_slice(slice_idx):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    sl = np.clip(slice_idx, 0, n_slices - 1)
    ct_slice = ct_norm[:, :, sl]
    mask_slice = mask_vol[:, :, sl]
    axes[0].imshow(ct_slice.T, cmap="gray", origin="lower")
    axes[0].set_title(f"CT — slice {sl}")
    axes[0].axis("off")
    axes[1].imshow(ct_slice.T, cmap="gray", origin="lower")
    mask_plot = np.ma.masked_where(mask_slice == 0, mask_slice)
    axes[1].imshow(mask_plot.T, cmap="nipy_spectral", alpha=0.5, origin="lower")
    axes[1].set_title(f"CT + mask overlay — slice {sl}")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

widgets.interact(show_slice, slice_idx=widgets.IntSlider(min=0, max=n_slices - 1, value=n_slices // 2, description="Slice"))

## 4. Optional: grid of all slices (thumbnails)

In [ ]:
# Show a grid of every Nth slice to get an overview (adjust step)
step = 10  # e.g. slices 0, 10, 20, ...
indices = list(range(0, n_slices, step))
n_plot = len(indices)
n_cols = 5
n_rows = (n_plot + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
axes = axes.flatten()
for i, sl in enumerate(indices):
    ax = axes[i]
    ct_slice = ct_norm[:, :, sl]
    mask_slice = mask_vol[:, :, sl]
    ax.imshow(ct_slice.T, cmap="gray", origin="lower")
    mask_plot = np.ma.masked_where(mask_slice == 0, mask_slice)
    ax.imshow(mask_plot.T, cmap="nipy_spectral", alpha=0.5, origin="lower")
    ax.set_title(f"{sl}")
    ax.axis("off")
for j in range(i + 1, len(axes)):
    axes[j].axis("off")
plt.suptitle(f"CT + mask overlay — every {step}th slice")
plt.tight_layout()
plt.show()